# KoELECTRA-NER Dataset Generator

이 노트북은 **한 파일만으로 독립 실행**되도록 구성되어 있습니다.

구성:
- 공통 유틸리티
- 공통 업무 문맥 풀
- ENTITY 8종: 각 타입별 별도 코드셀
- NON_ENTITY 8종: 각 타입별 별도 코드셀
- 전체 생성 및 검증 코드셀

출력 컬럼:
`id,text,entity_type,entity_text,start,end,label`

출력 위치:
`datasets/generated_entities_ipynb/`

In [ ]:
# 공통 유틸리티
from pathlib import Path
from dataclasses import dataclass, asdict
import csv, random, re

ROWS_PER_FILE = 5000
RANDOM_SEED = 20260514
random.seed(RANDOM_SEED)

OUT_DIR = (Path.cwd() / 'datasets' / 'generated_entities_ipynb').resolve()
OUT_DIR.mkdir(parents=True, exist_ok=True)
print('OUT_DIR =', OUT_DIR)

@dataclass(frozen=True)
class Row:
    id: str
    text: str
    entity_type: str
    entity_text: str
    start: int
    end: int
    label: str

def pick(xs):
    return random.choice(xs)

def chance(p):
    return random.random() < p

def token(n, chars):
    return ''.join(random.choice(chars) for _ in range(n))

def render(template, **values):
    for k, v in values.items():
        template = template.replace('{' + k + '}', str(v))
    return template

def has_batchim(text):
    if not text:
        return False
    code = ord(text[-1])
    return 0xAC00 <= code <= 0xD7A3 and (code - 0xAC00) % 28 != 0

def particle(text, pair):
    b = has_batchim(text)
    if pair == '이/가': return '이' if b else '가'
    if pair == '은/는': return '은' if b else '는'
    if pair == '을/를': return '을' if b else '를'
    if pair == '와/과': return '과' if b else '와'
    if pair == '으로/로':
        code = ord(text[-1])
        jong = (code - 0xAC00) % 28 if 0xAC00 <= code <= 0xD7A3 else 0
        return '으로' if jong not in (0, 8) else '로'
    raise ValueError(pair)

def wp(text, pair):
    return text + particle(text, pair)

def polish(text):
    text = text.replace('PMO은', 'PMO는')
    text = re.sub(r'([0-9])를', r'\\1을', text)
    text = re.sub(r'([0-9])가', r'\\1이', text)
    text = re.sub(r'([0-9])는', r'\\1은', text)
    return text.strip()

BAD_PATTERNS = re.compile(r'층로|층를|그룹와|그룹로|원가 포함|사용료은|처리 처리|요청 요청|전환 전환|운영 운영전환|메모 메모|까지까지|[{}]|건는|시간가|시간로|사번는|PMO은')

def make_row(entity_type, label, idx, text, entity_text):
    text = polish(text)
    start = text.find(entity_text)
    if start < 0:
        raise ValueError(f'entity not found: {entity_type} / {entity_text} / {text}')
    if text.find(entity_text, start + len(entity_text)) >= 0:
        raise ValueError(f'ambiguous entity: {entity_type} / {entity_text} / {text}')
    end = start + len(entity_text)
    if text[start:end] != entity_text:
        raise ValueError(f'span mismatch: {entity_type} / {entity_text} / {text}')
    prefix = entity_type.lower() + ('_non_entity_' if label == 'NON_ENTITY' else '_')
    return Row(prefix + str(idx).zfill(6), text, entity_type, entity_text, start, end, label)

def generate_rows(entity_type, label, generator, global_texts=None):
    if global_texts is None:
        global_texts = set()
    rows, local_texts, local_entities = [], set(), set()
    attempts = 0
    while len(rows) < ROWS_PER_FILE:
        attempts += 1
        if attempts > ROWS_PER_FILE * 1000:
            raise RuntimeError(f'too many attempts: {entity_type} {label}')
        text, entity = generator(len(rows) + 1)
        text = polish(text)
        if len(text) < 16 or text == entity or BAD_PATTERNS.search(text):
            continue
        if text in local_texts or text in global_texts or entity in local_entities:
            continue
        row = make_row(entity_type, label, len(rows) + 1, text, entity)
        rows.append(row)
        local_texts.add(row.text)
        local_entities.add(row.entity_text)
        global_texts.add(row.text)
    return rows

def write_csv(path, rows):
    with path.open('w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=['id','text','entity_type','entity_text','start','end','label'])
        writer.writeheader()
        for row in rows:
            writer.writerow(asdict(row))

def validate_rows(rows):
    return {
        'rows': len(rows),
        'unique_ids': len({r.id for r in rows}),
        'unique_texts': len({r.text for r in rows}),
        'unique_entities': len({r.entity_text for r in rows}),
        'bad_spans': sum(r.text[r.start:r.end] != r.entity_text for r in rows),
        'bad_patterns': sum(bool(BAD_PATTERNS.search(r.text)) for r in rows),
    }


In [ ]:
# 공통 업무 문맥 풀
TEAMS = ['보안팀','인프라팀','플랫폼팀','클라우드운영팀','데이터팀','개발팀','QA팀','서비스기획팀','PMO','고객성공팀','CS팀','영업팀','재무팀','법무팀','구매팀','계약관리팀','감사실','마케팅팀','SRE팀','DBA그룹','교육지원팀','파트너운영팀']
CHANNELS = ['메일','슬랙','팀즈','결재함','운영 로그','Jira 티켓','회의록','감사 대응표','보안 점검표','구매 요청서','정산 메모','배포 노트','온보딩 체크리스트']
SYSTEMS = ['ERP','CRM','SSO','VPN','GitLab','Jenkins','Datadog','Zendesk','Confluence','SAP','Tableau','Snowflake','Kubernetes','Okta','ServiceNow','Redmine']
TIME_WORDS = ['오늘 오전','오늘 오후','금일 중','내일 오전','이번 주','다음 주','월말까지','분기 마감 전','배포 전','계약 체결 전','정산 마감 전','감사 대응 중','회의 전','외부 발송 전']


## ENTITY 생성기 8종

In [ ]:
# PERSON ENTITY: 사람 실명. 직함은 문장에 넣되 span은 이름만 잡는다.
PERSON_SURNAMES = list('김이박최정강조윤장임한오서신권황안송류홍전고문양손배백허유남심노하곽성차주우구민진엄채원천방공현함변염여추도석소선설마길연위표명')
PERSON_GIVEN_A = list('민서지하도유준시현수예채은다아태연성재윤가나원우동승혜보규진영혁솔라린빈호율리찬건인정세로해봄온별담겸환희경슬주산소')
PERSON_GIVEN_B = list('우아준윤현서민영진희호원빈율연경수정하림결찬겸온솔리별재훈혁미린담인주완석비안은나라이채후열')
PERSON_ENTITY_TEMPLATES = [
    '{TEAM}은 {E}에게 접근 권한 검토 의견을 요청했습니다.',
    '{CHANNEL}에서 {E} 담당 건이 승인 대기 상태로 남아 있습니다.',
    '{E} 담당자가 고객 문의를 접수하고 후속 조치를 등록했습니다.',
    '{SYSTEM} 감사 로그에서 {E} 이름의 다운로드 기록을 확인했습니다.',
    '외부 발송 전 {E} 관리자의 승인 여부를 확인해야 합니다.',
    '{TEAM} 주간보고에 {E}의 처리 내역을 반영했습니다.',
    '{CHANNEL} 코멘트에 따르면 {E} 팀장이 최종 확인자입니다.',
]
def gen_person_entity(i):
    while True:
        e = pick(PERSON_SURNAMES) + pick(PERSON_GIVEN_A) + pick(PERSON_GIVEN_B)
        if e[-1] not in '은는이가을를의와과에':
            break
    return render(pick(PERSON_ENTITY_TEMPLATES), E=e, TEAM=pick(TEAMS), CHANNEL=pick(CHANNELS), SYSTEM=pick(SYSTEMS)), e


In [ ]:
# ADDRESS ENTITY: 도로명/지번/건물/층/호수 포함 주소.
ADDRESS_AREAS = ['서울시 강남구','서울시 서초구','서울시 마포구','부산시 해운대구','대구시 수성구','인천시 연수구','대전시 유성구','경기도 성남시 분당구','경기도 수원시 영통구','제주도 제주시']
ADDRESS_ROADS = ['테헤란로','판교역로','센텀중앙로','디지털로','중앙대로','송도과학로','혁신대로','첨단과기로','가산디지털로','월드컵북로']
ADDRESS_DONGS = ['역삼동','서초동','상암동','성수동','잠실동','송도동','정자동','마두동','삼평동','청담동']
ADDRESS_BUILDINGS = ['스마트허브','센터필드','파크타워','테크노밸리','메트로타워','그린빌딩','이노베이션타워','비즈니스센터']
ADDRESS_ENTITY_TEMPLATES = [
    '{TEAM} 현장 방문지는 {E}로 등록되어 있습니다.',
    '보안 점검 대상 사이트를 {E}로 지정했습니다.',
    '계약서상 사업장 소재지는 {E}입니다.',
    '{CHANNEL}에 올라온 배송지 변경 주소는 {E}입니다.',
    '온사이트 지원 위치는 {E}이며 출입 등록이 필요합니다.',
]
def gen_address_entity(i):
    area, road, dong, bld = pick(ADDRESS_AREAS), pick(ADDRESS_ROADS), pick(ADDRESS_DONGS), pick(ADDRESS_BUILDINGS)
    if chance(.4): e = f'{area} {road} {i}-{random.randint(1,80)}'
    elif chance(.5): e = f'{area} {road} {random.randint(1,900)} {bld} {random.randint(101,2600)}호'
    else: e = f'{area} {dong} {random.randint(10,999)} {bld} {random.randint(1,12)}동 {random.randint(101,2505)}호'
    return render(pick(ADDRESS_ENTITY_TEMPLATES), E=e, TEAM=pick(TEAMS), CHANNEL=pick(CHANNELS)), e


In [ ]:
# PASSWORD ENTITY: 비밀번호/임시 암호/초기 PW/서비스 계정 암호.
PASSWORD_CHARS = 'ABCDEFGHJKLMNPQRSTUVWXYZabcdefghijkmnopqrstuvwxyz23456789'
PASSWORD_SPECIALS = '!@#$%^&*_-+=?'
PASSWORD_ENTITY_TEMPLATES = [
    '임시 비밀번호는 {E}입니다.',
    '초기 로그인 PW: {E}',
    '서비스 계정 비밀번호를 {E}로 전달받았습니다.',
    '{CHANNEL}에 공유된 임시 암호 {E}를 삭제해 주세요.',
    '관리자 계정의 초기 암호는 {E}입니다.',
]
def gen_password_entity(i):
    if chance(.25): e = str(random.randint(10000000,99999999))
    elif chance(.5): e = token(random.randint(9,18), PASSWORD_CHARS)
    else: e = token(7, PASSWORD_CHARS) + pick(list(PASSWORD_SPECIALS)) + token(6, PASSWORD_CHARS) + str(random.randint(10,99))
    return render(pick(PASSWORD_ENTITY_TEMPLATES), E=e, CHANNEL=pick(CHANNELS)), e


In [ ]:
# API_KEY ENTITY: API key/token/secret/credential.
API_CHARS = 'ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz0123456789_-'
API_ENTITY_TEMPLATES = [
    'API 호출 시 헤더에 X-API-KEY: {E}를 추가하세요.',
    '신규 API 키 {E}가 운영 콘솔에서 발급되었습니다.',
    '배치 서버 환경변수 API_TOKEN 값은 {E}입니다.',
    '{TEAM}에서 {E} 토큰의 만료일을 확인하고 있습니다.',
    '웹훅 서명 검증용 secret은 {E}입니다.',
]
def gen_api_key_entity(i):
    if chance(.25): e = 'sk_live_' + token(38, API_CHARS)
    elif chance(.35): e = 'ghp_' + token(36, API_CHARS)
    elif chance(.5): e = 'AKIA' + token(16, 'ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789')
    else: e = token(random.randint(36,52), API_CHARS)
    return render(pick(API_ENTITY_TEMPLATES), E=e, TEAM=pick(TEAMS)), e


In [ ]:
# DOCUMENT ENTITY: 문서명/파일명/계약명/제안서명/설정 파일명.
DOCUMENT_SUBJECTS = ['유지보수','라이선스','보안점검','개인정보처리','데이터이관','클라우드전환','정산','입찰','비밀유지','하도급','검수완료']
DOCUMENT_KINDS = ['계약서','협약서','제안서','사업계획서','검토보고서','회의록','정산내역서','보안검토서','인수인계서','MOU']
DOCUMENT_EXTS = ['docx','hwp','hwpx','pdf','pptx','xlsx','doc']
DOCUMENT_ENTITY_TEMPLATES = [
    '{E} 파일을 검토해 주세요.',
    '첨부된 {E}의 접근 권한을 요청합니다.',
    '외부 공유 전 {E}를 마스킹해야 합니다.',
    '고객사에 전달할 {E}을 준비했습니다.',
    '서버에서 {E} 열람 이력이 확인되었습니다.',
]
def gen_document_entity(i):
    e = f"{random.randint(2023,2027)}년 {pick(DOCUMENT_SUBJECTS)} {pick(DOCUMENT_KINDS)} {pick(['초안','최종본','검토본','개정본'])} {i}.{pick(DOCUMENT_EXTS)}"
    return render(pick(DOCUMENT_ENTITY_TEMPLATES), E=e), e


In [ ]:
# ORG ENTITY: 외부 회사/고객사/공급사/협력사.
ORG_PREFIXES = ['새벽','한빛','에이스','블루','그린','프라임','넥스트','코어','데이터','오로라','하이퍼','모던']
ORG_DOMAINS = ['테크','소프트','네트웍스','시스템즈','솔루션','로지스','파이낸스','클라우드','팩토리','링크']
ORG_SUFFIXES = ['주식회사','유한회사','코리아','Labs','Partners','Networks','그룹','컴퍼니']
ORG_ENTITY_TEMPLATES = [
    '계약사는 {E}로 등록되어 있습니다.',
    '신규 고객사 {E}의 온보딩 일정을 확정했습니다.',
    '공급사 {E}와 유지보수 조건을 재협의합니다.',
    '{CHANNEL}에 {E} 관련 계약 변경 요청이 올라왔습니다.',
    '{E}의 보안 심사 결과를 공유해 주세요.',
]
def gen_org_entity(i):
    e = f'{pick(ORG_PREFIXES)}{pick(ORG_DOMAINS)}{i} {pick(ORG_SUFFIXES)}'
    return render(pick(ORG_ENTITY_TEMPLATES), E=e, CHANNEL=pick(CHANNELS)), e


In [ ]:
# PROJECT ENTITY: 프로젝트명/과제명/코드명.
PROJECT_DOMAINS = ['결제','보안','검색','CRM','ERP','데이터레이크','인증','클라우드','정산','고객경험','파트너 API','개인정보 파기']
PROJECT_ACTIONS = ['고도화','전환','개편','연동','자동화','통합','최적화','재구축','표준화']
PROJECT_CODES = ['Alpha','Nova','Glacier','Lynx','Atlas','Pulse','Vector','Orion','Nimbus']
PROJECT_ENTITY_TEMPLATES = [
    '{E} 킥오프 회의가 내일 오전으로 잡혔습니다.',
    '{E}의 WBS를 최신 일정으로 업데이트했습니다.',
    '주간 보고서에 {E} 진행률을 추가했습니다.',
    '{E} 관련 리스크를 PMO에 보고했습니다.',
    '{CHANNEL}에 {E} 관련 일정 변경 요청이 등록되었습니다.',
]
def gen_project_entity(i):
    if chance(.35): e = f'{pick(PROJECT_CODES)}-Q{random.randint(1,4)}-{i}'
    else: e = f"{pick(PROJECT_DOMAINS)} {pick(PROJECT_ACTIONS)} {pick(['Phase1','Phase2','2차','3차','파일럿'])}-{i}"
    return render(pick(PROJECT_ENTITY_TEMPLATES), E=e, CHANNEL=pick(CHANNELS)), e


In [ ]:
# FINANCIAL ENTITY: 금액/비율/재무 수치.
FIN_MONEY_CONTEXTS = ['계약 금액','정산 금액','입찰가','월 운영비','장비 단가','환급액','라이선스 비용','클라우드 사용료']
FIN_RATE_CONTEXTS = ['마진율','전환율','예산 집행률','납기 준수율','할인율','성장률']
FINANCIAL_ENTITY_TEMPLATES = [
    '이번 안건의 {C}은 {E}입니다.',
    '회의록 기준 {C}이 {E}로 확정되었습니다.',
    '견적서에 기재된 {C}: {E}',
    '{TEAM}은 {C}을 {E}로 다시 산정했습니다.',
    '분기 보고서의 {C}은 {E}로 집계되었습니다.',
]
def gen_financial_entity(i):
    if chance(.3): e, c = f'{random.randint(1,980)}억 {random.randint(1,9)}천만원', pick(FIN_MONEY_CONTEXTS)
    elif chance(.45): e, c = f'{random.randint(1,99)}.{random.randint(0,9)}%', pick(FIN_RATE_CONTEXTS)
    else: e, c = f'{random.randint(100,999999):,}원', pick(FIN_MONEY_CONTEXTS)
    return render(pick(FINANCIAL_ENTITY_TEMPLATES), E=e, C=c, TEAM=pick(TEAMS)), e


## NON_ENTITY 생성기 8종

In [ ]:
# PERSON NON_ENTITY: 실명이 아닌 역할명/공용 담당자/그룹 라벨.
PERSON_NON_ROLES = ['담당자','관리자','승인자','검토자','온콜 담당','계약 담당','현장 담당','배포 담당']
PERSON_NON_TEMPLATES = [
    '{E} 역할은 아직 개인 이름으로 배정되지 않았습니다.',
    '{CHANNEL}에는 {E}만 표시되고 실제 담당자 이름은 비공개 처리되었습니다.',
    '고객 회신란에는 {E}라고만 적혀 있어 실명 확인이 필요합니다.',
    '{TEAM}은 {E} 직책 기준으로 결재선을 구성했습니다.',
]
def gen_person_non(i):
    e = f"{pick(TEAMS)} {pick(PERSON_NON_ROLES)} {pick(['A','B','C','1조','2조','공용'])}-{i:04d}"
    return render(pick(PERSON_NON_TEMPLATES), E=e, TEAM=pick(TEAMS), CHANNEL=pick(CHANNELS)), e


In [ ]:
# ADDRESS NON_ENTITY: 정확한 주소가 아닌 권역/장소명/회의실/위치 태그.
ADDRESS_NON_PLACES = ['강남','판교','상암','송도','여의도','성수','대전','부산','회의실 A','대회의실','서버실','옥상 테라스','1층 로비','현장 사무소']
ADDRESS_NON_KINDS = ['권역','상권','담당 구역','배송 권역','예약','대기','점검','교육']
ADDRESS_NON_TEMPLATES = [
    '{E} 근처에서 고객 미팅이 예정되어 있습니다.',
    '{CHANNEL}에는 {E}까지만 적혀 있어 정확한 주소가 아닙니다.',
    '행사 장소는 {E}로 표시되었지만 상세 주소는 별도 공지 예정입니다.',
    '{SYSTEM} 위치 태그가 {E}로 저장되었습니다.',
]
def gen_address_non(i):
    e = f'{pick(ADDRESS_NON_PLACES)} {pick(ADDRESS_NON_KINDS)}-{i:04d}'
    return render(pick(ADDRESS_NON_TEMPLATES), E=e, CHANNEL=pick(CHANNELS), SYSTEM=pick(SYSTEMS)), e


In [ ]:
# PASSWORD NON_ENTITY: 비밀번호처럼 보일 수 있으나 주문번호/운송장/사번/쿠폰 등인 값.
PASSWORD_NON_KINDS = ['주문번호','운송장 번호','장비 시리얼','사번','쿠폰 코드','접수 번호','전화번호 끝자리','자산 태그']
PASSWORD_NON_PREFIXES = ['OD','SO','TRK','AS','EQ','RMA','TKT','SN']
PASSWORD_NON_TEMPLATES = [
    '{E}는 {K}라서 비밀번호로 사용하면 안 됩니다.',
    '{CHANNEL}에 공유된 {K} {E}는 로그인 암호가 아닙니다.',
    '{K} {E}가 접수되었지만 패스워드 변경 대상은 아닙니다.',
    '{SYSTEM} 화면의 {E} 값은 {K} 필드에 저장됩니다.',
]
def gen_password_non(i):
    k = pick(PASSWORD_NON_KINDS)
    e = str(random.randint(1000,9999)) if '끝자리' in k else f'{pick(PASSWORD_NON_PREFIXES)}-{random.randint(100000,999999)}-{token(3,"ABCDEFGHJKLMNPQRSTUVWXYZ")}'
    return render(pick(PASSWORD_NON_TEMPLATES), E=e, K=k, CHANNEL=pick(CHANNELS), SYSTEM=pick(SYSTEMS)), e


In [ ]:
# API_KEY NON_ENTITY: 실제 인증 키가 아닌 추적 ID/더미 토큰/체크섬/배포 태그.
API_NON_KINDS = ['추적 ID','요청 ID','상관관계 ID','쿠폰 번호','빌드 해시','배포 태그','샘플 토큰','마스킹 예시값']
API_NON_PREFIXES = ['REQ','TRACE','LOG','CASE','SAMPLE','MASK']
API_NON_TEMPLATES = [
    '{E}는 {K}이며 인증용 API 키가 아닙니다.',
    '{CHANNEL}에 남은 {K} {E}는 호출 권한을 부여하지 않습니다.',
    '문서 예시에 포함된 {E}는 실제 토큰이 아닌 더미 값입니다.',
    '{TEAM}은 {K} {E}로 로그를 추적했습니다.',
]
def gen_api_key_non(i):
    k = pick(API_NON_KINDS)
    e = f'{pick(API_NON_PREFIXES)}-{token(8,"ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789")}-{i % 100:02d}'
    return render(pick(API_NON_TEMPLATES), E=e, K=k, TEAM=pick(TEAMS), CHANNEL=pick(CHANNELS)), e


In [ ]:
# DOCUMENT NON_ENTITY: 업무 문서명이 아닌 공개 템플릿/샘플/예시 파일.
DOCUMENT_NON_NAMES = ['README','LICENSE','CHANGELOG','sample','template','example','placeholder','SUPPORT','NOTICE','styleguide']
DOCUMENT_NON_TAGS = ['v1','v2','blank','public','sample','guide']
DOCUMENT_NON_EXTS = ['md','txt','pdf','docx','json','yaml','pptx','xlsx']
DOCUMENT_NON_TEMPLATES = [
    '{E}는 공개 템플릿 파일이라 계약 문서로 분류하지 않습니다.',
    '교육 자료에는 {E}를 예시 파일로 첨부했습니다.',
    '{CHANNEL}에 올라온 {E}는 샘플 문서입니다.',
    '{TEAM}은 {E}를 참고 양식으로만 사용했습니다.',
]
def gen_document_non(i):
    e = f'{pick(DOCUMENT_NON_NAMES)}_{pick(DOCUMENT_NON_TAGS)}_{i}.{pick(DOCUMENT_NON_EXTS)}'
    return render(pick(DOCUMENT_NON_TEMPLATES), E=e, TEAM=pick(TEAMS), CHANNEL=pick(CHANNELS)), e


In [ ]:
# ORG NON_ENTITY: 외부 조직명이 아닌 내부 부서/도구/협업 채널명.
ORG_NON_GROUPS = ['보안정책팀','클라우드운영팀','플랫폼개발팀','PMO','SRE셀','DBA그룹','정산운영팀','감사대응TF']
ORG_NON_TOOLS = ['Slack','Teams','Jira','Confluence','GitLab','Figma','Notion','Tableau','Snowflake','Postman']
ORG_NON_TEMPLATES = [
    '{E}은 내부 부서명이라 외부 조직명으로 보지 않습니다.',
    '{CHANNEL}에는 {E} 담당으로만 표시되어 고객사명이 아닙니다.',
    '계약 상대방이 아니라 {E}에서 접수한 내부 요청입니다.',
    '{TEAM}은 {E} 항목을 내부 협업 채널로 분류했습니다.',
]
def gen_org_non(i):
    e = f'{pick(ORG_NON_GROUPS if chance(.6) else ORG_NON_TOOLS)} {pick(["공용","운영","검토","승인","보드","채널"])}-{i:04d}'
    return render(pick(ORG_NON_TEMPLATES), E=e, TEAM=pick(TEAMS), CHANNEL=pick(CHANNELS)), e


In [ ]:
# PROJECT NON_ENTITY: 프로젝트명이 아닌 반복 업무명/운영 태스크/워크플로 단계.
PROJECT_NON_TASKS = ['정기 점검','업무 인수인계','권한 검토','운영 이관','최종 검토','회의 준비','자료 취합','로그 확인','백업 점검','월간 보고','공지 발송']
PROJECT_NON_TAGS = ['1차','2차','오전','오후','월간','분기','임시','정기']
PROJECT_NON_TEMPLATES = [
    '{E}은 반복 업무명이라 공식 프로젝트명으로 분류하지 않습니다.',
    '{CHANNEL}에 등록된 {E} 일정은 프로젝트 코드가 아닙니다.',
    '{E}은 담당자 업무 상태라 프로젝트명 마스킹 대상이 아닙니다.',
    '{SYSTEM} 티켓의 {E} 항목은 워크플로 단계명입니다.',
]
def gen_project_non(i):
    e = f'{pick(PROJECT_NON_TASKS)} {pick(PROJECT_NON_TAGS)}-{i:04d}'
    return render(pick(PROJECT_NON_TEMPLATES), E=e, CHANNEL=pick(CHANNELS), SYSTEM=pick(SYSTEMS)), e


In [ ]:
# FINANCIAL NON_ENTITY: 금액이 아닌 건수/시간/차수/버전/좌석번호.
FIN_NON_CONTEXTS = ['대기 건수','로그 라인 수','참석 인원','알림 횟수','대여 수량','교육 시간','회의 차수','버전 번호','테스트 케이스 수']
FIN_NON_TEMPLATES = [
    '{C}는 {E}로 집계되었지만 금액 정보는 아닙니다.',
    '{CHANNEL}에 기록된 {E}는 {C} 값입니다.',
    '보고서의 {E}는 단순 수량이라 재무 수치로 보지 않습니다.',
    '{SYSTEM} 화면에는 {C}가 {E}로 표시됩니다.',
]
def gen_financial_non(i):
    c = pick(FIN_NON_CONTEXTS)
    if '시간' in c: e = f'{i}시간'
    elif '차수' in c: e = f'{i}차'
    elif '버전' in c: e = f'v{random.randint(1,9)}.{random.randint(0,20)}.{i}'
    else: e = f'{i:,}건'
    return render(pick(FIN_NON_TEMPLATES), E=e, C=c, CHANNEL=pick(CHANNELS), SYSTEM=pick(SYSTEMS)), e


## 전체 실행 및 검증

In [ ]:
# 16개 파일 생성
GENERATORS = [
    ('PERSON','ENTITY',gen_person_entity),
    ('ADDRESS','ENTITY',gen_address_entity),
    ('PASSWORD','ENTITY',gen_password_entity),
    ('API_KEY','ENTITY',gen_api_key_entity),
    ('DOCUMENT','ENTITY',gen_document_entity),
    ('ORG','ENTITY',gen_org_entity),
    ('PROJECT','ENTITY',gen_project_entity),
    ('FINANCIAL','ENTITY',gen_financial_entity),
    ('PERSON','NON_ENTITY',gen_person_non),
    ('ADDRESS','NON_ENTITY',gen_address_non),
    ('PASSWORD','NON_ENTITY',gen_password_non),
    ('API_KEY','NON_ENTITY',gen_api_key_non),
    ('DOCUMENT','NON_ENTITY',gen_document_non),
    ('ORG','NON_ENTITY',gen_org_non),
    ('PROJECT','NON_ENTITY',gen_project_non),
    ('FINANCIAL','NON_ENTITY',gen_financial_non),
]

global_texts = set()
all_rows = []
summaries = []

for entity_type, label, generator in GENERATORS:
    random.seed(f'{RANDOM_SEED}:{entity_type}:{label}')
    rows = generate_rows(entity_type, label, generator, global_texts)
    suffix = 'entity' if label == 'ENTITY' else 'non_entity'
    path = OUT_DIR / f'{entity_type.lower()}_{suffix}_5000.csv'
    write_csv(path, rows)
    summaries.append((path.name, validate_rows(rows)))
    all_rows.extend(rows)

total = len(all_rows)
global_unique_texts = len({r.text for r in all_rows})
global_unique_ids = len({r.id for r in all_rows})
bad_spans = sum(r.text[r.start:r.end] != r.entity_text for r in all_rows)
bad_patterns = sum(bool(BAD_PATTERNS.search(r.text)) for r in all_rows)

print('OUT_DIR:', OUT_DIR)
print('total_rows:', total)
print('global_unique_texts:', global_unique_texts)
print('global_duplicate_texts:', total - global_unique_texts)
print('global_duplicate_ids:', total - global_unique_ids)
print('bad_spans:', bad_spans)
print('bad_patterns:', bad_patterns)
print('\nper-file summary')
for filename, summary in summaries:
    print(filename, summary)

assert total == 80000
assert global_unique_texts == total
assert global_unique_ids == total
assert bad_spans == 0
assert bad_patterns == 0
